# HW3: Named Entity Recognition with BiLSTM
### CSCI 544 - Spring 2026
**Name:** Sohail Haresh Gidwani
**USC ID:** 7321203258

In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import numpy as np
import os, random, subprocess, shutil

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

dev = torch.device('cuda' if torch.cuda.is_available() else
                   'mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
                   else 'cpu')
print('Using:', dev)

Using: mps


## Data Loading and Exploration

In [15]:
def load_sents(fpath, labeled=True):
    sents, cur = [], []
    with open(fpath) as f:
        for ln in f:
            ln = ln.rstrip('\n')
            if ln.strip() == '':
                if cur:
                    sents.append(cur)
                    cur = []
                continue
            parts = ln.split()
            idx = int(parts[0])
            word = parts[1]
            tag = parts[2] if labeled and len(parts) >= 3 else None
            cur.append((idx, word, tag))
    if cur:
        sents.append(cur)
    return sents

def build_w2i(sents):
    wc = {}
    for s in sents:
        for _, w, _ in s:
            wc[w] = wc.get(w, 0) + 1
    w2i = {'<pad>': 0, '<unk>': 1}
    for w in sorted(wc):
        w2i[w] = len(w2i)
    return w2i

def build_t2i(sents):
    tags = set()
    for s in sents:
        for _, _, t in s:
            if t: tags.add(t)
    return {t: i for i, t in enumerate(sorted(tags))}

In [16]:
tr_sents = load_sents('data/train')
dv_sents = load_sents('data/dev')
ts_sents = load_sents('data/test', labeled=False)

w2i = build_w2i(tr_sents)
t2i = build_t2i(tr_sents)
i2t = {v: k for k, v in t2i.items()}

print('train: %d sentences' % len(tr_sents))
print('dev:   %d sentences' % len(dv_sents))
print('test:  %d sentences' % len(ts_sents))
print('vocab size:', len(w2i))
print('tag set:', t2i)

train: 14987 sentences
dev:   3466 sentences
test:  3684 sentences
vocab size: 23626
tag set: {'B-LOC': 0, 'B-MISC': 1, 'B-ORG': 2, 'B-PER': 3, 'I-LOC': 4, 'I-MISC': 5, 'I-ORG': 6, 'I-PER': 7, 'O': 8}


In [17]:
# check a few samples
for s in tr_sents[:2]:
    print(' '.join('%s/%s' % (w, t) for _, w, t in s))
    print()

# tag counts
from collections import Counter
tag_counts = Counter()
for s in tr_sents:
    for _, _, t in s:
        tag_counts[t] += 1
print('tag distribution:')
for tag, cnt in tag_counts.most_common():
    print('  %-8s %d' % (tag, cnt))

EU/B-ORG rejects/O German/B-MISC call/O to/O boycott/O British/B-MISC lamb/O ./O

Peter/B-PER Blackburn/I-PER

tag distribution:
  O        170524
  B-LOC    7140
  B-PER    6600
  B-ORG    6321
  I-PER    4528
  I-ORG    3704
  B-MISC   3438
  I-LOC    1157
  I-MISC   1155


In [18]:
def write_output(pred_sents, fpath):
    with open(fpath, 'w') as f:
        for sent in pred_sents:
            for idx, word, tag in sent:
                f.write('{} {} {}\n'.format(idx, word, tag))
            f.write('\n')

# copy eval script to working dir so perl can find it
if not os.path.exists('conll03eval'):
    shutil.copy('eval/conll03eval', 'conll03eval')

def run_eval(pred_file, gold_file):
    result = subprocess.run(
        ['python3', 'eval/eval.py', '-p', pred_file, '-g', gold_file],
        capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)

## Task 1: Simple BiLSTM (40 pts)

Architecture: `Embedding -> BiLSTM -> Linear -> ELU -> Classifier`

Fixed hyperparameters from the assignment:
- embedding dim = 100
- LSTM layers = 1, hidden dim = 256, dropout = 0.33
- linear output dim = 128

Tuned hyperparameters: lr=0.15, momentum=0.9, batch_size=32, epochs=30, StepLR(step=7, gamma=0.5), gradient clip=5.0

In [19]:
class NERData(Dataset):
    def __init__(self, sents, w2i, t2i=None):
        self.items = []
        unk = w2i['<unk>']
        for s in sents:
            wids = [w2i.get(w, unk) for _, w, _ in s]
            tids = [t2i[t] for _, _, t in s] if (t2i and s[0][2] is not None) else None
            raw = [(i, w) for i, w, _ in s]
            self.items.append((wids, tids, raw))

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def collate_ner(batch):
    wids_l, tids_l, raw_l = zip(*batch)
    lens = torch.tensor([len(w) for w in wids_l])
    wids = pad_sequence([torch.tensor(w, dtype=torch.long) for w in wids_l],
                        batch_first=True, padding_value=0)
    if tids_l[0] is not None:
        tids = pad_sequence([torch.tensor(t, dtype=torch.long) for t in tids_l],
                            batch_first=True, padding_value=-1)
    else:
        tids = None
    return wids, tids, lens, raw_l

In [20]:
class NERTagger(nn.Module):
    def __init__(self, vsz, n_tags, emb_dim=100, hid=256, fc_dim=128, drop=0.33):
        super().__init__()
        self.emb = nn.Embedding(vsz, emb_dim, padding_idx=0)
        nn.init.uniform_(self.emb.weight, -0.05, 0.05)
        self.emb.weight.data[0].fill_(0)
        self.drop = nn.Dropout(drop)
        self.lstm = nn.LSTM(emb_dim, hid, num_layers=1, batch_first=True,
                            bidirectional=True, dropout=drop)
        self.fc = nn.Linear(hid * 2, fc_dim)
        self.elu = nn.ELU()
        self.clf = nn.Linear(fc_dim, n_tags)

    def forward(self, x, lengths):
        e = self.drop(self.emb(x))
        pk = pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.lstm(pk)
        h, _ = pad_packed_sequence(h, batch_first=True)
        h = self.drop(h)
        return self.clf(self.elu(self.fc(h)))

In [21]:
def train_epoch(tagger, loader, opt, crit, clip=5.0):
    tagger.train()
    total_loss, nb = 0, 0
    for wids, tids, lens, _ in loader:
        wids, tids = wids.to(dev), tids.to(dev)
        opt.zero_grad()
        logits = tagger(wids, lens.to(dev))
        loss = crit(logits.view(-1, logits.size(-1)), tids.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(tagger.parameters(), clip)
        opt.step()
        total_loss += loss.item()
        nb += 1
    return total_loss / nb

@torch.no_grad()
def eval_model(tagger, loader, crit):
    tagger.eval()
    total, nb, correct, total_tok = 0, 0, 0, 0
    for wids, tids, lens, _ in loader:
        wids, tids = wids.to(dev), tids.to(dev)
        logits = tagger(wids, lens.to(dev))
        loss = crit(logits.view(-1, logits.size(-1)), tids.view(-1))
        total += loss.item()
        nb += 1
        preds = logits.argmax(-1)
        mask = tids != -1
        correct += ((preds == tids) & mask).sum().item()
        total_tok += mask.sum().item()
    return total / nb, correct / total_tok if total_tok > 0 else 0

@torch.no_grad()
def predict_tags(tagger, loader):
    tagger.eval()
    all_sents = []
    for wids, _, lens, raw_batch in loader:
        wids = wids.to(dev)
        logits = tagger(wids, lens.to(dev))
        preds = logits.argmax(-1).cpu()
        for b in range(len(lens)):
            slen = lens[b].item()
            sent = []
            for j in range(slen):
                idx, word = raw_batch[b][j]
                tag = i2t[preds[b][j].item()]
                sent.append((idx, word, tag))
            all_sents.append(sent)
    return all_sents

In [22]:
train_dl = DataLoader(NERData(tr_sents, w2i, t2i), batch_size=32,
                      shuffle=True, collate_fn=collate_ner)
dev_dl = DataLoader(NERData(dv_sents, w2i, t2i), batch_size=32,
                    shuffle=False, collate_fn=collate_ner)

tagger1 = NERTagger(len(w2i), len(t2i)).to(dev)
opt1 = optim.SGD(tagger1.parameters(), lr=0.15, momentum=0.9)
sched1 = optim.lr_scheduler.StepLR(opt1, step_size=7, gamma=0.5)
crit = nn.CrossEntropyLoss(ignore_index=-1)

print(tagger1)
total_params = sum(p.numel() for p in tagger1.parameters())
print('total params:', total_params)

NERTagger(
  (emb): Embedding(23626, 100, padding_idx=0)
  (drop): Dropout(p=0.33, inplace=False)
  (lstm): LSTM(100, 256, batch_first=True, dropout=0.33, bidirectional=True)
  (fc): Linear(in_features=512, out_features=128, bias=True)
  (elu): ELU(alpha=1.0)
  (clf): Linear(in_features=128, out_features=9, bias=True)
)
total params: 3162609


/Users/sohail/miniconda3/lib/python3.13/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.33 and num_layers=1
  warnings.warn(


In [23]:
best_loss = 1e9
for ep in range(1, 31):
    tr = train_epoch(tagger1, train_dl, opt1, crit)
    dv, acc = eval_model(tagger1, dev_dl, crit)
    lr_now = sched1.get_last_lr()[0]
    print('ep %02d | train %.4f | dev %.4f | acc %.4f | lr %.5f' % (ep, tr, dv, acc, lr_now))
    sched1.step()
    if dv < best_loss:
        best_loss = dv
        torch.save({'model': tagger1.state_dict(), 'w2i': w2i, 't2i': t2i},
                   'blstm1.pt')
        print('  -> saved best')

ep 01 | train 0.7483 | dev 0.6219 | acc 0.8373 | lr 0.15000
  -> saved best
ep 02 | train 0.4154 | dev 0.3761 | acc 0.8977 | lr 0.15000
  -> saved best
ep 03 | train 0.2398 | dev 0.3029 | acc 0.9258 | lr 0.15000
  -> saved best
ep 04 | train 0.1653 | dev 0.2369 | acc 0.9407 | lr 0.15000
  -> saved best
ep 05 | train 0.1165 | dev 0.2340 | acc 0.9470 | lr 0.15000
  -> saved best
ep 06 | train 0.0809 | dev 0.2059 | acc 0.9539 | lr 0.15000
  -> saved best
ep 07 | train 0.0610 | dev 0.2000 | acc 0.9561 | lr 0.15000
  -> saved best
ep 08 | train 0.0406 | dev 0.2031 | acc 0.9580 | lr 0.07500
ep 09 | train 0.0343 | dev 0.1985 | acc 0.9598 | lr 0.07500
  -> saved best
ep 10 | train 0.0302 | dev 0.2083 | acc 0.9599 | lr 0.07500
ep 11 | train 0.0269 | dev 0.2329 | acc 0.9572 | lr 0.07500
ep 12 | train 0.0241 | dev 0.2117 | acc 0.9596 | lr 0.07500
ep 13 | train 0.0219 | dev 0.2278 | acc 0.9604 | lr 0.07500
ep 14 | train 0.0201 | dev 0.2376 | acc 0.9584 | lr 0.07500
ep 15 | train 0.0151 | dev 0.230

In [24]:
# load best checkpoint
ckpt1 = torch.load('blstm1.pt', weights_only=False, map_location=dev)
tagger1.load_state_dict(ckpt1['model'])

# dev predictions
dev_preds = predict_tags(tagger1, dev_dl)
write_output(dev_preds, 'dev1.out')
print('wrote dev1.out (%d sentences)' % len(dev_preds))

# test predictions
test_dl = DataLoader(NERData(ts_sents, w2i), batch_size=32,
                     shuffle=False, collate_fn=collate_ner)
test_preds = predict_tags(tagger1, test_dl)
write_output(test_preds, 'test1.out')
print('wrote test1.out (%d sentences)' % len(test_preds))

wrote dev1.out (3466 sentences)
wrote test1.out (3684 sentences)


### Task 1 Evaluation

In [25]:
run_eval('dev1.out', 'data/dev')

processed 51578 tokens with 5942 phrases; found: 5548 phrases; correct: 4532.
accuracy:  95.98%; precision:  81.69%; recall:  76.27%; FB1:  78.89
              LOC: precision:  92.29%; recall:  82.09%; FB1:  86.89  1634
             MISC: precision:  85.04%; recall:  75.81%; FB1:  80.16  822
              ORG: precision:  74.77%; recall:  71.59%; FB1:  73.14  1284
              PER: precision:  75.50%; recall:  74.10%; FB1:  74.79  1808



## Task 2: BiLSTM + GloVe + Case Features (60 pts)

Same BiLSTM architecture as Task 1 but initialized with GloVe 100d embeddings.

**Case sensitivity strategy:** GloVe is case-insensitive, so all words are looked up by their lowercase form. To keep capitalization info, I assign each word a case category (lowercase, UPPERCASE, Titlecase, has_digit, other) and map it through a learned 10-dim embedding. This gets concatenated with the 100-dim word embedding, giving the LSTM an input of 110.

**Vocab expansion:** I add dev/test words that have GloVe vectors to the vocabulary so the model can use pretrained representations for them instead of falling back to UNK.

**Differential learning rate:** The pretrained word embeddings use lr=0.001 so they don't drift too far from GloVe, while the rest of the model (LSTM, linear, case embeddings) uses lr=0.1 to learn quickly.

Tuned: emb_lr=0.001, other_lr=0.1, momentum=0.9, batch_size=32, epochs=50, StepLR(step=10, gamma=0.5)

In [26]:
def load_glove(fpath):
    print('loading glove...')
    vecs = {}
    with open(fpath, encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split(' ')
            word = parts[0]
            vec = np.array(parts[1:], dtype=np.float32)
            vecs[word] = vec
    print('loaded %d vectors, dim=%d' % (len(vecs), len(next(iter(vecs.values())))))
    return vecs

def make_emb_matrix(w2i, glove, emb_dim=100):
    n = len(w2i)
    mat = np.random.uniform(-0.05, 0.05, (n, emb_dim)).astype(np.float32)
    mat[0] = 0  # pad
    found = 0
    for w, idx in w2i.items():
        if idx < 2: continue
        wl = w.lower()
        if wl in glove:
            mat[idx] = glove[wl]
            found += 1
        elif w in glove:
            mat[idx] = glove[w]
            found += 1
    print('coverage: %d / %d (%.1f%%)' % (found, n, 100*found/n))
    return mat

In [27]:
N_CASES = 6   # 0=pad, 1=lower, 2=UPPER, 3=Title, 4=digit, 5=other
CASE_DIM = 10

def get_case(w):
    if any(c.isdigit() for c in w): return 4
    if w.islower(): return 1
    if w.isupper(): return 2
    if w[0].isupper(): return 3
    return 5

class NERData2(Dataset):
    def __init__(self, sents, w2i, t2i=None):
        self.items = []
        unk = w2i['<unk>']
        for s in sents:
            wids = [w2i.get(w, unk) for _, w, _ in s]
            cids = [get_case(w) for _, w, _ in s]
            tids = [t2i[t] for _, _, t in s] if (t2i and s[0][2] is not None) else None
            raw = [(i, w) for i, w, _ in s]
            self.items.append((wids, cids, tids, raw))

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def collate_ner2(batch):
    wids_l, cids_l, tids_l, raw_l = zip(*batch)
    lens = torch.tensor([len(w) for w in wids_l])
    wids = pad_sequence([torch.tensor(w, dtype=torch.long) for w in wids_l],
                        batch_first=True, padding_value=0)
    cids = pad_sequence([torch.tensor(c, dtype=torch.long) for c in cids_l],
                        batch_first=True, padding_value=0)
    if tids_l[0] is not None:
        tids = pad_sequence([torch.tensor(t, dtype=torch.long) for t in tids_l],
                            batch_first=True, padding_value=-1)
    else:
        tids = None
    return wids, cids, tids, lens, raw_l

In [28]:
class GloVeNERTagger(nn.Module):
    def __init__(self, vsz, n_tags, emb_matrix,
                 n_cases=N_CASES, case_dim=CASE_DIM,
                 hid=256, fc_dim=128, drop=0.33):
        super().__init__()
        emb_dim = emb_matrix.shape[1]
        self.word_emb = nn.Embedding(vsz, emb_dim, padding_idx=0)
        self.word_emb.weight.data.copy_(torch.from_numpy(emb_matrix))
        self.case_emb = nn.Embedding(n_cases, case_dim, padding_idx=0)
        lstm_in = emb_dim + case_dim
        self.drop = nn.Dropout(drop)
        self.lstm = nn.LSTM(lstm_in, hid, num_layers=1, batch_first=True,
                            bidirectional=True, dropout=drop)
        self.fc = nn.Linear(hid * 2, fc_dim)
        self.elu = nn.ELU()
        self.clf = nn.Linear(fc_dim, n_tags)

    def forward(self, wids, cids, lengths):
        we = self.word_emb(wids)
        ce = self.case_emb(cids)
        x = torch.cat([we, ce], dim=-1)
        x = self.drop(x)
        pk = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.lstm(pk)
        h, _ = pad_packed_sequence(h, batch_first=True)
        h = self.drop(h)
        return self.clf(self.elu(self.fc(h)))

In [29]:
def train_epoch2(tagger, loader, opt, crit, clip=5.0):
    tagger.train()
    total_loss, nb = 0, 0
    for wids, cids, tids, lens, _ in loader:
        wids, cids, tids = wids.to(dev), cids.to(dev), tids.to(dev)
        opt.zero_grad()
        logits = tagger(wids, cids, lens.to(dev))
        loss = crit(logits.view(-1, logits.size(-1)), tids.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(tagger.parameters(), clip)
        opt.step()
        total_loss += loss.item()
        nb += 1
    return total_loss / nb

@torch.no_grad()
def eval_model2(tagger, loader, crit):
    tagger.eval()
    total, nb, correct, total_tok = 0, 0, 0, 0
    for wids, cids, tids, lens, _ in loader:
        wids, cids, tids = wids.to(dev), cids.to(dev), tids.to(dev)
        logits = tagger(wids, cids, lens.to(dev))
        loss = crit(logits.view(-1, logits.size(-1)), tids.view(-1))
        total += loss.item()
        nb += 1
        preds = logits.argmax(-1)
        mask = tids != -1
        correct += ((preds == tids) & mask).sum().item()
        total_tok += mask.sum().item()
    return total / nb, correct / total_tok if total_tok > 0 else 0

@torch.no_grad()
def predict_tags2(tagger, loader):
    tagger.eval()
    all_sents = []
    for wids, cids, _, lens, raw_batch in loader:
        wids, cids = wids.to(dev), cids.to(dev)
        logits = tagger(wids, cids, lens.to(dev))
        preds = logits.argmax(-1).cpu()
        for b in range(len(lens)):
            slen = lens[b].item()
            sent = []
            for j in range(slen):
                idx, word = raw_batch[b][j]
                tag = i2t[preds[b][j].item()]
                sent.append((idx, word, tag))
            all_sents.append(sent)
    return all_sents

In [30]:
glove = load_glove('glove.6B.100d')

# expand vocab: add dev/test words that have GloVe vectors
extra = 0
for sents in [dv_sents, ts_sents]:
    for s in sents:
        for _, w, _ in s:
            if w not in w2i and w.lower() in glove:
                w2i[w] = len(w2i)
                extra += 1
# rebuild reverse tag map (unchanged) but note new vocab size
i2t = {v: k for k, v in t2i.items()}
print('expanded vocab by %d -> %d total' % (extra, len(w2i)))

emb_mat = make_emb_matrix(w2i, glove, emb_dim=100)
del glove

loading glove...
loaded 400000 vectors, dim=100
expanded vocab by 5331 -> 28957 total
coverage: 26340 / 28957 (91.0%)


In [31]:
train_dl2 = DataLoader(NERData2(tr_sents, w2i, t2i), batch_size=32,
                       shuffle=True, collate_fn=collate_ner2)
dev_dl2 = DataLoader(NERData2(dv_sents, w2i, t2i), batch_size=32,
                     shuffle=False, collate_fn=collate_ner2)

tagger2 = GloVeNERTagger(len(w2i), len(t2i), emb_mat).to(dev)

# differential lr: small for pretrained embeddings, larger for rest
emb_params = list(tagger2.word_emb.parameters())
emb_ids = set(id(p) for p in emb_params)
other_params = [p for p in tagger2.parameters() if id(p) not in emb_ids]

opt2 = optim.SGD([
    {'params': emb_params, 'lr': 0.001},
    {'params': other_params, 'lr': 0.1}
], momentum=0.9)
sched2 = optim.lr_scheduler.StepLR(opt2, step_size=10, gamma=0.5)

print(tagger2)
total_params = sum(p.numel() for p in tagger2.parameters())
print('total params:', total_params)

GloVeNERTagger(
  (word_emb): Embedding(28957, 100, padding_idx=0)
  (case_emb): Embedding(6, 10, padding_idx=0)
  (drop): Dropout(p=0.33, inplace=False)
  (lstm): LSTM(110, 256, batch_first=True, dropout=0.33, bidirectional=True)
  (fc): Linear(in_features=512, out_features=128, bias=True)
  (elu): ELU(alpha=1.0)
  (clf): Linear(in_features=128, out_features=9, bias=True)
)
total params: 3716249


/Users/sohail/miniconda3/lib/python3.13/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.33 and num_layers=1
  warnings.warn(


In [32]:
best_loss2 = 1e9
for ep in range(1, 51):
    tr = train_epoch2(tagger2, train_dl2, opt2, crit)
    dv, acc = eval_model2(tagger2, dev_dl2, crit)
    lr_now = sched2.get_last_lr()[0]
    print('ep %02d | train %.4f | dev %.4f | acc %.4f | lr %.5f' % (ep, tr, dv, acc, lr_now))
    sched2.step()
    if dv < best_loss2:
        best_loss2 = dv
        torch.save({'model': tagger2.state_dict(), 'w2i': w2i, 't2i': t2i,
                     'n_cases': N_CASES, 'case_dim': CASE_DIM},
                   'blstm2.pt')
        print('  -> saved best')

ep 01 | train 0.2765 | dev 0.1312 | acc 0.9654 | lr 0.00100
  -> saved best
ep 02 | train 0.1415 | dev 0.0985 | acc 0.9726 | lr 0.00100
  -> saved best
ep 03 | train 0.1174 | dev 0.0845 | acc 0.9765 | lr 0.00100
  -> saved best
ep 04 | train 0.1042 | dev 0.0876 | acc 0.9745 | lr 0.00100
ep 05 | train 0.0957 | dev 0.0720 | acc 0.9796 | lr 0.00100
  -> saved best
ep 06 | train 0.0883 | dev 0.0720 | acc 0.9798 | lr 0.00100
ep 07 | train 0.0826 | dev 0.0655 | acc 0.9816 | lr 0.00100
  -> saved best
ep 08 | train 0.0782 | dev 0.0641 | acc 0.9813 | lr 0.00100
  -> saved best
ep 09 | train 0.0745 | dev 0.0596 | acc 0.9825 | lr 0.00100
  -> saved best
ep 10 | train 0.0718 | dev 0.0578 | acc 0.9832 | lr 0.00100
  -> saved best
ep 11 | train 0.0668 | dev 0.0517 | acc 0.9847 | lr 0.00050
  -> saved best
ep 12 | train 0.0627 | dev 0.0514 | acc 0.9846 | lr 0.00050
  -> saved best
ep 13 | train 0.0615 | dev 0.0504 | acc 0.9845 | lr 0.00050
  -> saved best
ep 14 | train 0.0605 | dev 0.0492 | acc 0.98

In [33]:
ckpt2 = torch.load('blstm2.pt', weights_only=False, map_location=dev)
tagger2.load_state_dict(ckpt2['model'])

dev_preds2 = predict_tags2(tagger2, dev_dl2)
write_output(dev_preds2, 'dev2.out')
print('wrote dev2.out (%d sentences)' % len(dev_preds2))

test_dl2 = DataLoader(NERData2(ts_sents, w2i), batch_size=64,
                      shuffle=False, collate_fn=collate_ner2)
test_preds2 = predict_tags2(tagger2, test_dl2)
write_output(test_preds2, 'test2.out')
print('wrote test2.out (%d sentences)' % len(test_preds2))

wrote dev2.out (3466 sentences)
wrote test2.out (3684 sentences)


### Task 2 Evaluation

In [34]:
run_eval('dev2.out', 'data/dev')

processed 51578 tokens with 5942 phrases; found: 6032 phrases; correct: 5543.
accuracy:  98.76%; precision:  91.89%; recall:  93.29%; FB1:  92.58
              LOC: precision:  94.62%; recall:  95.75%; FB1:  95.18  1859
             MISC: precision:  85.59%; recall:  86.98%; FB1:  86.28  937
              ORG: precision:  87.93%; recall:  89.11%; FB1:  88.52  1359
              PER: precision:  95.21%; recall:  97.01%; FB1:  96.10  1877



## Bonus: BiLSTM-CNN (10 + 10 pts)

Adding a character-level CNN on top of Task 2. Each word gets processed character by character through a CNN, max-pooled to a fixed size, then concatenated with the word + case embeddings before going into the BiLSTM.

- char embedding dim = 30 (required)
- CNN: 1 layer, kernel_size=3, 50 filters
- LSTM input = 100 (glove) + 10 (case) + 50 (char CNN) = 160

In [35]:
# build character vocab from training data
char_set = set()
for s in tr_sents:
    for _, w, _ in s:
        for c in w:
            char_set.add(c)
c2i = {'<pad>': 0, '<unk>': 1}
for c in sorted(char_set):
    c2i[c] = len(c2i)
print('char vocab:', len(c2i))

class CharCNN(nn.Module):
    def __init__(self, n_chars, char_emb=30, n_filters=50, kernel=3):
        super().__init__()
        self.cemb = nn.Embedding(n_chars, char_emb, padding_idx=0)
        self.conv = nn.Conv1d(char_emb, n_filters, kernel, padding=kernel//2)
        self.n_filters = n_filters

    def forward(self, char_ids):
        # char_ids: (N, max_word_len)
        e = self.cemb(char_ids)         # (N, max_wl, char_emb)
        e = e.transpose(1, 2)           # (N, char_emb, max_wl)
        c = self.conv(e)                # (N, n_filters, max_wl)
        c = c.max(dim=2)[0]             # (N, n_filters)
        return c

char vocab: 86


In [36]:
class NERTaggerCNN(nn.Module):
    def __init__(self, vsz, n_tags, emb_matrix, n_chars,
                 n_cases=N_CASES, case_dim=CASE_DIM,
                 char_emb=30, n_filters=50, kernel=3,
                 hid=256, fc_dim=128, drop=0.33):
        super().__init__()
        emb_dim = emb_matrix.shape[1]
        self.word_emb = nn.Embedding(vsz, emb_dim, padding_idx=0)
        self.word_emb.weight.data.copy_(torch.from_numpy(emb_matrix))
        self.case_emb = nn.Embedding(n_cases, case_dim, padding_idx=0)
        self.char_cnn = CharCNN(n_chars, char_emb, n_filters, kernel)

        lstm_in = emb_dim + case_dim + n_filters
        self.drop = nn.Dropout(drop)
        self.lstm = nn.LSTM(lstm_in, hid, num_layers=1, batch_first=True,
                            bidirectional=True, dropout=drop)
        self.fc = nn.Linear(hid * 2, fc_dim)
        self.elu = nn.ELU()
        self.clf = nn.Linear(fc_dim, n_tags)

    def forward(self, wids, cids, char_ids, lengths):
        B, S = wids.shape
        we = self.word_emb(wids)
        ce = self.case_emb(cids)

        # char CNN: reshape to (B*S, max_word_len), apply, reshape back
        char_flat = char_ids.view(B * S, -1)
        char_feat = self.char_cnn(char_flat)
        char_feat = char_feat.view(B, S, -1)

        x = torch.cat([we, ce, char_feat], dim=-1)
        x = self.drop(x)
        pk = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.lstm(pk)
        h, _ = pad_packed_sequence(h, batch_first=True)
        h = self.drop(h)
        return self.clf(self.elu(self.fc(h)))

In [37]:
class NERDataCNN(Dataset):
    def __init__(self, sents, w2i, c2i, t2i=None):
        self.items = []
        unk_w = w2i['<unk>']
        unk_c = c2i['<unk>']
        for s in sents:
            wids = [w2i.get(w, unk_w) for _, w, _ in s]
            cas = [get_case(w) for _, w, _ in s]
            chars = [[c2i.get(c, unk_c) for c in w] for _, w, _ in s]
            tids = [t2i[t] for _, _, t in s] if (t2i and s[0][2] is not None) else None
            raw = [(i, w) for i, w, _ in s]
            self.items.append((wids, cas, chars, tids, raw))

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def collate_cnn(batch):
    wids_l, cids_l, chars_l, tids_l, raw_l = zip(*batch)
    lens = torch.tensor([len(w) for w in wids_l])
    max_slen = max(lens).item()

    wids = pad_sequence([torch.tensor(w, dtype=torch.long) for w in wids_l],
                        batch_first=True, padding_value=0)
    cids = pad_sequence([torch.tensor(c, dtype=torch.long) for c in cids_l],
                        batch_first=True, padding_value=0)

    # pad char sequences: (batch, max_slen, max_word_len)
    max_wlen = max(len(c) for chars in chars_l for c in chars)
    char_pad = torch.zeros(len(batch), max_slen, max_wlen, dtype=torch.long)
    for b, chars in enumerate(chars_l):
        for w, cseq in enumerate(chars):
            char_pad[b, w, :len(cseq)] = torch.tensor(cseq, dtype=torch.long)

    if tids_l[0] is not None:
        tids = pad_sequence([torch.tensor(t, dtype=torch.long) for t in tids_l],
                            batch_first=True, padding_value=-1)
    else:
        tids = None
    return wids, cids, char_pad, tids, lens, raw_l

In [38]:
def train_epoch_cnn(tagger, loader, opt, crit, clip=5.0):
    tagger.train()
    total_loss, nb = 0, 0
    for wids, cids, char_ids, tids, lens, _ in loader:
        wids = wids.to(dev)
        cids = cids.to(dev)
        char_ids = char_ids.to(dev)
        tids = tids.to(dev)
        opt.zero_grad()
        logits = tagger(wids, cids, char_ids, lens.to(dev))
        loss = crit(logits.view(-1, logits.size(-1)), tids.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(tagger.parameters(), clip)
        opt.step()
        total_loss += loss.item()
        nb += 1
    return total_loss / nb

@torch.no_grad()
def eval_model_cnn(tagger, loader, crit):
    tagger.eval()
    total, nb, correct, total_tok = 0, 0, 0, 0
    for wids, cids, char_ids, tids, lens, _ in loader:
        wids = wids.to(dev)
        cids = cids.to(dev)
        char_ids = char_ids.to(dev)
        tids = tids.to(dev)
        logits = tagger(wids, cids, char_ids, lens.to(dev))
        loss = crit(logits.view(-1, logits.size(-1)), tids.view(-1))
        total += loss.item()
        nb += 1
        preds = logits.argmax(-1)
        mask = tids != -1
        correct += ((preds == tids) & mask).sum().item()
        total_tok += mask.sum().item()
    return total / nb, correct / total_tok if total_tok > 0 else 0

@torch.no_grad()
def predict_tags_cnn(tagger, loader):
    tagger.eval()
    all_sents = []
    for wids, cids, char_ids, _, lens, raw_batch in loader:
        wids = wids.to(dev)
        cids = cids.to(dev)
        char_ids = char_ids.to(dev)
        logits = tagger(wids, cids, char_ids, lens.to(dev))
        preds = logits.argmax(-1).cpu()
        for b in range(len(lens)):
            slen = lens[b].item()
            sent = []
            for j in range(slen):
                idx, word = raw_batch[b][j]
                tag = i2t[preds[b][j].item()]
                sent.append((idx, word, tag))
            all_sents.append(sent)
    return all_sents

In [39]:
# reload glove embedding matrix (we deleted the raw dict earlier)
glove_reload = load_glove('glove.6B.100d')
emb_mat2 = make_emb_matrix(w2i, glove_reload, emb_dim=100)
del glove_reload

train_dl_cnn = DataLoader(NERDataCNN(tr_sents, w2i, c2i, t2i), batch_size=32,
                          shuffle=True, collate_fn=collate_cnn)
dev_dl_cnn = DataLoader(NERDataCNN(dv_sents, w2i, c2i, t2i), batch_size=32,
                        shuffle=False, collate_fn=collate_cnn)

tagger_cnn = NERTaggerCNN(len(w2i), len(t2i), emb_mat2, len(c2i)).to(dev)

# differential lr same as task 2
emb_p = list(tagger_cnn.word_emb.parameters())
emb_ids_cnn = set(id(p) for p in emb_p)
other_p = [p for p in tagger_cnn.parameters() if id(p) not in emb_ids_cnn]

opt_cnn = optim.SGD([
    {'params': emb_p, 'lr': 0.001},
    {'params': other_p, 'lr': 0.1}
], momentum=0.9)
sched_cnn = optim.lr_scheduler.StepLR(opt_cnn, step_size=10, gamma=0.5)

print(tagger_cnn)

loading glove...
loaded 400000 vectors, dim=100
coverage: 26340 / 28957 (91.0%)
NERTaggerCNN(
  (word_emb): Embedding(28957, 100, padding_idx=0)
  (case_emb): Embedding(6, 10, padding_idx=0)
  (char_cnn): CharCNN(
    (cemb): Embedding(86, 30, padding_idx=0)
    (conv): Conv1d(30, 50, kernel_size=(3,), stride=(1,), padding=(1,))
  )
  (drop): Dropout(p=0.33, inplace=False)
  (lstm): LSTM(160, 256, batch_first=True, dropout=0.33, bidirectional=True)
  (fc): Linear(in_features=512, out_features=128, bias=True)
  (elu): ELU(alpha=1.0)
  (clf): Linear(in_features=128, out_features=9, bias=True)
)


/Users/sohail/miniconda3/lib/python3.13/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.33 and num_layers=1
  warnings.warn(


In [40]:
best_loss_cnn = 1e9
for ep in range(1, 51):
    tr = train_epoch_cnn(tagger_cnn, train_dl_cnn, opt_cnn, crit)
    dv, acc = eval_model_cnn(tagger_cnn, dev_dl_cnn, crit)
    lr_now = sched_cnn.get_last_lr()[0]
    print('ep %02d | train %.4f | dev %.4f | acc %.4f | lr %.5f' % (ep, tr, dv, acc, lr_now))
    sched_cnn.step()
    if dv < best_loss_cnn:
        best_loss_cnn = dv
        torch.save({'model': tagger_cnn.state_dict(), 'w2i': w2i, 't2i': t2i,
                     'c2i': c2i}, 'blstm_cnn.pt')
        print('  -> saved best')

ep 01 | train 0.2777 | dev 0.1205 | acc 0.9681 | lr 0.00100
  -> saved best
ep 02 | train 0.1259 | dev 0.0909 | acc 0.9749 | lr 0.00100
  -> saved best
ep 03 | train 0.1046 | dev 0.0767 | acc 0.9791 | lr 0.00100
  -> saved best
ep 04 | train 0.0907 | dev 0.0773 | acc 0.9779 | lr 0.00100
ep 05 | train 0.0830 | dev 0.0615 | acc 0.9821 | lr 0.00100
  -> saved best
ep 06 | train 0.0755 | dev 0.0629 | acc 0.9820 | lr 0.00100
ep 07 | train 0.0696 | dev 0.0582 | acc 0.9834 | lr 0.00100
  -> saved best
ep 08 | train 0.0650 | dev 0.0545 | acc 0.9842 | lr 0.00100
  -> saved best
ep 09 | train 0.0604 | dev 0.0586 | acc 0.9838 | lr 0.00100
ep 10 | train 0.0580 | dev 0.0534 | acc 0.9844 | lr 0.00100
  -> saved best
ep 11 | train 0.0498 | dev 0.0491 | acc 0.9859 | lr 0.00050
  -> saved best
ep 12 | train 0.0479 | dev 0.0485 | acc 0.9857 | lr 0.00050
  -> saved best
ep 13 | train 0.0460 | dev 0.0485 | acc 0.9858 | lr 0.00050
ep 14 | train 0.0451 | dev 0.0465 | acc 0.9865 | lr 0.00050
  -> saved best


In [41]:
ckpt_cnn = torch.load('blstm_cnn.pt', weights_only=False, map_location=dev)
tagger_cnn.load_state_dict(ckpt_cnn['model'])

# dev eval
dev_preds_cnn = predict_tags_cnn(tagger_cnn, dev_dl_cnn)
write_output(dev_preds_cnn, 'dev_cnn.out')
print('wrote dev_cnn.out')
run_eval('dev_cnn.out', 'data/dev')

# test predictions for bonus
test_dl_cnn = DataLoader(NERDataCNN(ts_sents, w2i, c2i), batch_size=64,
                         shuffle=False, collate_fn=collate_cnn)
test_preds_cnn = predict_tags_cnn(tagger_cnn, test_dl_cnn)
write_output(test_preds_cnn, 'pred')
print('wrote pred (%d sentences)' % len(test_preds_cnn))

wrote dev_cnn.out
processed 51578 tokens with 5942 phrases; found: 6037 phrases; correct: 5529.
accuracy:  98.73%; precision:  91.59%; recall:  93.05%; FB1:  92.31
              LOC: precision:  94.39%; recall:  95.32%; FB1:  94.85  1855
             MISC: precision:  86.24%; recall:  86.33%; FB1:  86.29  923
              ORG: precision:  86.42%; recall:  89.71%; FB1:  88.04  1392
              PER: precision:  95.29%; recall:  96.58%; FB1:  95.93  1867

wrote pred (3684 sentences)


## Results Summary

In [42]:
print('=== Task 1: Simple BiLSTM ===')
run_eval('dev1.out', 'data/dev')
print()
print('=== Task 2: BiLSTM + GloVe + Case ===')
run_eval('dev2.out', 'data/dev')
print()
print('=== Bonus: BiLSTM-CNN ===')
run_eval('dev_cnn.out', 'data/dev')

=== Task 1: Simple BiLSTM ===
processed 51578 tokens with 5942 phrases; found: 5548 phrases; correct: 4532.
accuracy:  95.98%; precision:  81.69%; recall:  76.27%; FB1:  78.89
              LOC: precision:  92.29%; recall:  82.09%; FB1:  86.89  1634
             MISC: precision:  85.04%; recall:  75.81%; FB1:  80.16  822
              ORG: precision:  74.77%; recall:  71.59%; FB1:  73.14  1284
              PER: precision:  75.50%; recall:  74.10%; FB1:  74.79  1808


=== Task 2: BiLSTM + GloVe + Case ===
processed 51578 tokens with 5942 phrases; found: 6032 phrases; correct: 5543.
accuracy:  98.76%; precision:  91.89%; recall:  93.29%; FB1:  92.58
              LOC: precision:  94.62%; recall:  95.75%; FB1:  95.18  1859
             MISC: precision:  85.59%; recall:  86.98%; FB1:  86.28  937
              ORG: precision:  87.93%; recall:  89.11%; FB1:  88.52  1359
              PER: precision:  95.21%; recall:  97.01%; FB1:  96.10  1877


=== Bonus: BiLSTM-CNN ===
processed 51578 toke